# 03 - Preprocessing and Feature Preparation

**Owner:** Aliyan 

This notebook creates leakage-safe, reproducible representations for category classification and research-paper discovery. It validates the cleaned dataset, creates a stratified split, fits transformations on training data only, and saves split metadata and feature matrices.

## Reproducibility contract

- Input: `data/papers.parquet` produced by the cleaning notebook.
- Target: `primary_category`.
- Text: `text_raw` for TF-IDF and sentence-transformer embeddings.
- Numeric metadata: `year`, `month`, and abstract/title word counts.
- Split: stratified 80/20 train/test split with the shared seed.
- All learned transformations are fitted on the training partition only.
- Outputs are written to `results/` and use the same row order as the saved ids.

In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

TEST_SIZE = 0.20
REPO_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_DIR))
from src.config import SEED, EMBEDDING_MODEL, SAMPLE_SIZE

EMBEDDING_SAMPLE_SIZE = SAMPLE_SIZE
DATA_PATH = REPO_DIR / "data" / "papers.parquet"
RESULTS_DIR = REPO_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Expected cleaned dataset at {DATA_PATH}")
papers = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(papers):,} papers from {DATA_PATH}")

Loaded 119,919 papers from c:\Users\hp\OneDrive\Desktop\DDs_project\research_paper_discovery\data\papers.parquet


In [3]:
required = {"id", "text_raw", "primary_category", "year", "month", "title", "abstract"}
missing = required.difference(papers.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")
if papers["id"].duplicated().any():
    raise ValueError("Input ids must be unique")
if papers[["text_raw", "primary_category"]].isna().any().any():
    raise ValueError("Text and target columns must not contain nulls")
print("Target distribution:")
display(papers["primary_category"].value_counts().rename("count").to_frame())

Target distribution:


,count
primary_category,
cs.CL,39980
cs.CV,39973
cs.LG,39966


In [4]:
indices = np.arange(len(papers))
train_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=SEED,
    stratify=papers["primary_category"]
)
train = papers.iloc[train_idx].copy()
test = papers.iloc[test_idx].copy()
print(f"train={len(train):,}, test={len(test):,}")
print("train/test proportions:")
display(pd.concat([
    train["primary_category"].value_counts(normalize=True).rename("train"),
    test["primary_category"].value_counts(normalize=True).rename("test")
], axis=1))

split_metadata = {
    "seed": SEED,
    "test_size": TEST_SIZE,
    "train_ids": train["id"].tolist(),
    "test_ids": test["id"].tolist(),
}
(RESULTS_DIR / "split_metadata.json").write_text(
    json.dumps(split_metadata, indent=2), encoding="utf-8"
)

train=95,935, test=23,984
train/test proportions:


,train,test
primary_category,,
cs.CL,0.333392,0.333389
cs.CV,0.333330,0.333347
cs.LG,0.333278,0.333264


2158619

## TF-IDF word features

The vocabulary and inverse-document-frequency weights are learned from training text only. The resulting matrices are L2-normalized by the vectorizer.

In [5]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), min_df=2, max_df=0.95,
    sublinear_tf=True, norm="l2", dtype=np.float32
)
X_train_tfidf = tfidf.fit_transform(train["text_raw"])
X_test_tfidf = tfidf.transform(test["text_raw"])
y_train = train["primary_category"].to_numpy()
y_test = test["primary_category"].to_numpy()
print("TF-IDF shapes:", X_train_tfidf.shape, X_test_tfidf.shape)
from scipy.sparse import save_npz
save_npz(RESULTS_DIR / "tfidf_train.npz", X_train_tfidf)
save_npz(RESULTS_DIR / "tfidf_test.npz", X_test_tfidf)
np.save(RESULTS_DIR / "y_train.npy", y_train)
np.save(RESULTS_DIR / "y_test.npy", y_test)

TF-IDF shapes: (95935, 908488) (23984, 908488)


## Scaled numeric metadata

These features are derived from metadata already present in the cleaned dataset. Their means and standard deviations are learned from the training partition only.

In [6]:
def word_count(series):
    return series.astype(str).str.split().str.len().to_numpy(dtype=np.float32)

numeric_columns = ["year", "month", "title_words", "abstract_words"]
def numeric_features(frame):
    return np.column_stack([
        frame["year"].to_numpy(dtype=np.float32),
        frame["month"].to_numpy(dtype=np.float32),
        word_count(frame["title"]),
        word_count(frame["abstract"]),
    ])

scaler = StandardScaler()
X_train_numeric = scaler.fit_transform(numeric_features(train)).astype(np.float32)
X_test_numeric = scaler.transform(numeric_features(test)).astype(np.float32)
np.save(RESULTS_DIR / "numeric_train.npy", X_train_numeric)
np.save(RESULTS_DIR / "numeric_test.npy", X_test_numeric)
print("Numeric shapes:", X_train_numeric.shape, X_test_numeric.shape)

Numeric shapes: (95935, 4) (23984, 4)


## Sentence-transformer embeddings

Embeddings are generated for a deterministic stratified subset so the notebook remains practical on CPU. The model is loaded only when this section is run.

In [7]:
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer

embedding_n = min(EMBEDDING_SAMPLE_SIZE, len(papers))
embedding_idx, _ = train_test_split(
    np.arange(len(papers)), train_size=embedding_n, random_state=SEED,
    stratify=papers["primary_category"]
)
embedding_subset = papers.iloc[embedding_idx].sort_values("id").reset_index(drop=True)
encoder = SentenceTransformer(EMBEDDING_MODEL)
embeddings = encoder.encode(
    embedding_subset["text_raw"].tolist(), batch_size=32,
    show_progress_bar=True, normalize_embeddings=True,
    convert_to_numpy=True
).astype(np.float32)
np.save(RESULTS_DIR / "embeddings.npy", embeddings)
embedding_subset[["id", "primary_category"]].to_parquet(
    RESULTS_DIR / "embedding_ids.parquet", index=False
)
print("Embedding shape:", embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Embedding shape: (10000, 384)


In [8]:
manifest = {
    "seed": SEED,
    "embedding_model": EMBEDDING_MODEL,
    "embedding_sample_size": embedding_n,
    "tfidf_ngram_range": [1, 2],
    "tfidf_norm": "l2",
    "numeric_columns": numeric_columns,
    "train_rows": len(train),
    "test_rows": len(test),
    "tfidf_features": X_train_tfidf.shape[1],
    "embedding_dimension": int(embeddings.shape[1]),
}
(RESULTS_DIR / "feature_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
print(json.dumps(manifest, indent=2))

{
  "seed": 42,
  "embedding_model": "all-MiniLM-L6-v2",
  "embedding_sample_size": 10000,
  "tfidf_ngram_range": [
    1,
    2
  ],
  "tfidf_norm": "l2",
  "numeric_columns": [
    "year",
    "month",
    "title_words",
    "abstract_words"
  ],
  "train_rows": 95935,
  "test_rows": 23984,
  "tfidf_features": 908488,
  "embedding_dimension": 384
}
